In [7]:
import time
import random
import csv
import os

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# ==============================
# CONFIG
# ==============================

INDEX_URL = "https://www.indiabix.com/aptitude/questions-and-answers/"
CSV_FILE = "indiabix_full_aptitude_demo.csv"

MIN_DELAY = 2
MAX_DELAY = 4

visited_urls = set()
seen_questions = set()


# ==============================
# DELAY
# ==============================

def human_delay():
    time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))


# ==============================
# DRIVER
# ==============================

def create_driver():

    options = Options()

    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")

    driver = webdriver.Chrome(options=options)

    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )

    return driver


# ==============================
# CSV
# ==============================

def setup_csv():

    if not os.path.exists(CSV_FILE):

        with open(CSV_FILE, "w", newline="", encoding="utf-8") as f:

            writer = csv.writer(f)

            writer.writerow([
                "section",
                "page_number",
                "question",
                "option_a",
                "option_b",
                "option_c",
                "option_d",
                "correct_answer"
            ])


def save_row(row):

    with open(CSV_FILE, "a", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(row)


# ==============================
# PAGINATION
# ==============================

def go_next(driver):

    try:

        next_li = driver.find_element(
            By.XPATH,
            "//ul[contains(@class,'pagination')]//li[.//a[normalize-space()='Next']]"
        )

        if "disabled" in (next_li.get_attribute("class") or "").lower():
            return False

        next_btn = next_li.find_element(By.TAG_NAME, "a")

        driver.execute_script("arguments[0].click();", next_btn)

        time.sleep(3)

        return True

    except:
        return False


# ==============================
# GET TOPICS
# ==============================

def get_all_topics(driver):

    print("Fetching topic list...")

    driver.get(INDEX_URL)

    human_delay()

    topics = {}

    elements = driver.find_elements(
        By.XPATH,
        "//a[contains(@href, '/aptitude/') and not(contains(@href, '/questions-and-answers'))]"
    )

    for el in elements:

        name = (el.text or "").strip()

        url = el.get_attribute("href")

        if name and url:

            if name.lower() not in ["home", "aptitude"]:
                topics[name] = url

    return topics


# ==============================
# EXERCISES
# ==============================

def get_exercise_links(driver):

    exercises = {}

    try:

        links = driver.find_elements(
            By.XPATH,
            "//a[contains(@href,'/aptitude/') and contains(@href,'/')]"
        )

        for a in links:

            txt = (a.text or "").strip()

            href = a.get_attribute("href")

            if not txt or not href:
                continue

            if txt.lower() == "odd man out and series":
                continue

            if "exercise" in txt.lower() or "series" in txt.lower() or "numbers" in txt.lower():
                exercises[txt] = href

    except:
        pass

    return exercises


# ==============================
# SUB SECTIONS
# ==============================

def get_sub_sections(driver, topic_url, topic_name):

    driver.get(topic_url)

    human_delay()

    sub_sections = {}

    sub_sections[f"{topic_name} - General Questions"] = topic_url

    try:

        ds_links = driver.find_elements(
            By.XPATH,
            "//a[contains(text(),'Data Sufficiency')]"
        )

        for link in ds_links:

            name = (link.text or "").strip()

            href = link.get_attribute("href")

            if name and href:
                sub_sections[f"{topic_name} - {name}"] = href

    except:
        pass

    exercises = get_exercise_links(driver)

    for name, url in exercises.items():

        sub_sections[f"{topic_name} - {name}"] = url

    return sub_sections


# ==============================
# SCRAPER
# ==============================

def scrape_section(driver, section_name, url):

    global visited_urls
    global seen_questions

    if url in visited_urls:

        print("Skipping duplicate section:", section_name)
        return

    visited_urls.add(url)

    wait = WebDriverWait(driver, 15)

    page = 1

    driver.get(url)

    human_delay()

    while True:

        print(f"Scraping {section_name} → Page {page}")

        try:

            questions = wait.until(
                EC.presence_of_all_elements_located(
                    (By.CLASS_NAME, "bix-div-container")
                )
            )

        except:

            print("No questions found.")

            break

        for block in questions:

            try:

                question = block.find_element(
                    By.CLASS_NAME,
                    "bix-td-qtxt"
                ).text.strip()

            except:
                continue

            if question in seen_questions:
                continue

            seen_questions.add(question)

            option_rows = block.find_elements(
                By.CLASS_NAME,
                "bix-opt-row"
            )

            options = []

            correct_answer = ""

            for row in option_rows:

                try:

                    text = row.find_element(
                        By.CLASS_NAME,
                        "bix-td-option-val"
                    ).text.strip()

                except:

                    text = ""

                options.append(text)

                try:

                    btn = row.find_element(
                        By.CLASS_NAME,
                        "bix-td-option"
                    )

                    driver.execute_script(
                        "arguments[0].click();",
                        btn
                    )

                    time.sleep(0.5)

                    cls = row.find_element(
                        By.CLASS_NAME,
                        "bix-td-option-val"
                    ).get_attribute("class")

                    if "correct-ans" in cls:
                        correct_answer = text

                except:
                    pass

            while len(options) < 4:
                options.append("")

            save_row([
                section_name,
                page,
                question,
                options[0],
                options[1],
                options[2],
                options[3],
                correct_answer
            ])

        if not go_next(driver):
            break

        page += 1


# ==============================
# MAIN
# ==============================

def main():

    setup_csv()

    driver = create_driver()

    topics = get_all_topics(driver)

    print(f"\nFound {len(topics)} topics.\n")

    for topic_name, topic_url in topics.items():

        print("\nMapping Sub-sections for:", topic_name)

        sub_sections = get_sub_sections(
            driver,
            topic_url,
            topic_name
        )

        for sub_name, sub_url in sub_sections.items():

            try:

                scrape_section(
                    driver,
                    sub_name,
                    sub_url
                )

            except Exception as e:

                print("Error scraping:", sub_name, e)

    driver.quit()

    print("\nFinished scraping.")
    print("Saved to:", CSV_FILE)


# ==============================
# RUN
# ==============================

if __name__ == "__main__":
    main()

Fetching topic list...

Found 35 topics.


Mapping Sub-sections for: Problems on Trains
Scraping Problems on Trains - General Questions → Page 1
Scraping Problems on Trains - General Questions → Page 2
Scraping Problems on Trains - General Questions → Page 3
Scraping Problems on Trains - General Questions → Page 4
Scraping Problems on Trains - General Questions → Page 5
Scraping Problems on Trains - General Questions → Page 6
Scraping Problems on Trains - General Questions → Page 7
Scraping Problems on Trains - Problems on Trains - Data Sufficiency 1 → Page 1
Scraping Problems on Trains - Problems on Trains - Data Sufficiency 2 → Page 1
Scraping Problems on Trains - Problems on Trains - Data Sufficiency 3 → Page 1

Mapping Sub-sections for: Time and Distance
Scraping Time and Distance - General Questions → Page 1
Scraping Time and Distance - General Questions → Page 2
Scraping Time and Distance - General Questions → Page 3
Scraping Time and Distance - Time and Distance - Data Sufficien